# Training Dataset Audit

This notebook reproduces the dataset preparation path used by training in `/home/liondungl/projects/qcircuit-generation` and audits the results.

It focuses on the two places that matter for your suspicion:

- raw per-qubit dataset properties against the paper SRV setup
- the difference between `padding_mode=max` and `padding_mode=bucket`

In particular, it compares:

- the `z` metadata inferred for bucket mode from saved tensors
- the `z` that max-padding preprocessing would derive from actual gate occupancy
- the final prepared tensors and dataloader batch shapes

In [ ]:
import sys
from pathlib import Path
from pprint import pprint

from IPython.display import display

HELPER_DIR = Path('/home/liondungl/projects/master_thesis/notebooks')
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

from training_dataset_audit_helper import (
    DEFAULT_DATASET_ROOT,
    DEFAULT_TRAINING_CFG,
    build_loader_config,
    load_prepared_dataset,
    maybe_dataframe,
    preview_dataloaders,
    resolve_dataset_dirs,
    run_full_audit,
    summarize_prepared_dataset,
    summarize_raw_dataset,
)


In [ ]:
DATASET_ROOT = Path(DEFAULT_DATASET_ROOT)
TRAINING_CFG = Path(DEFAULT_TRAINING_CFG)
BATCH_SIZE = None
SPLIT_RATIO = 0.1

print('DATASET_ROOT =', DATASET_ROOT)
print('TRAINING_CFG =', TRAINING_CFG)
print('LOADER_CFG =', build_loader_config(TRAINING_CFG, batch_size=BATCH_SIZE))

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f'Dataset root does not exist: {DATASET_ROOT}\n'
        'Point DATASET_ROOT to your local copy of the official dataset.'
    )


In [ ]:
dataset_dirs = resolve_dataset_dirs(DATASET_ROOT)
raw_summaries = [summarize_raw_dataset(path) for path in dataset_dirs]

raw_table = []
for item in raw_summaries:
    raw_table.append({
        'dataset_dir': Path(item['dataset_dir']).name,
        'num_samples_saved': item['num_samples_saved'],
        'tensor_shape': item['tensor_shape'],
        'num_qubits_config': item['num_qubits_config'],
        'min_gates_config': item['min_gates_config'],
        'max_gates_config': item['max_gates_config'],
        'matches_paper_gate_pool': item['matches_paper_gate_pool'],
        'matches_paper_min_gates': item['matches_paper_min_gates'],
        'matches_paper_max_gates': item['matches_paper_max_gates'],
        'gate_length_min': item['gate_length_min'],
        'gate_length_median': item['gate_length_median'],
        'gate_length_max': item['gate_length_max'],
        'bucket_z_time_min': item['bucket_z_time_min'],
        'bucket_z_time_max': item['bucket_z_time_max'],
        'max_stage_z_time_min': item['max_stage_z_time_min'],
        'max_stage_z_time_max': item['max_stage_z_time_max'],
        'bucket_vs_max_time_mismatch': item['bucket_vs_max_time_mismatch'],
        'bucket_time_is_constant': item['bucket_time_is_constant'],
        'prompt_unique_count': item['prompt_unique_count'],
    })

display(maybe_dataframe(raw_table))


In [ ]:
for item in raw_summaries:
    name = Path(item['dataset_dir']).name
    print(f'\n=== {name} ===')
    print('Top prompts:')
    display(maybe_dataframe(item['top_prompts']))
    print('Entanglement buckets:')
    display(maybe_dataframe(item['entanglement_buckets']))
    print('Bucket-inferred z time counts:')
    display(maybe_dataframe(item['z_bucket_time_counts']))
    print('Max-stage z time counts:')
    display(maybe_dataframe(item['z_max_time_counts']))


In [ ]:
loader_max, max_dataset = load_prepared_dataset(
    DATASET_ROOT,
    training_cfg_path=TRAINING_CFG,
    padding_mode='max',
    batch_size=BATCH_SIZE,
)
loader_bucket, bucket_dataset = load_prepared_dataset(
    DATASET_ROOT,
    training_cfg_path=TRAINING_CFG,
    padding_mode='bucket',
    batch_size=BATCH_SIZE,
)

max_summary = summarize_prepared_dataset(max_dataset, padding_mode='max')
bucket_summary = summarize_prepared_dataset(bucket_dataset, padding_mode='bucket')

print('MAX PREPARATION')
pprint(max_summary)
print('\nBUCKET PREPARATION')
pprint(bucket_summary)


In [ ]:
effective_batch_size = build_loader_config(TRAINING_CFG, batch_size=BATCH_SIZE)['training']['batch_size']

preview_max = preview_dataloaders(
    loader_max,
    max_dataset,
    batch_size=effective_batch_size,
    split_ratio=SPLIT_RATIO,
)
preview_bucket = preview_dataloaders(
    loader_bucket,
    bucket_dataset,
    batch_size=effective_batch_size,
    split_ratio=SPLIT_RATIO,
)

print('MAX LOADER PREVIEW')
pprint(preview_max)
print('\nBUCKET LOADER PREVIEW')
pprint(preview_bucket)


In [ ]:
full_audit = run_full_audit(
    DATASET_ROOT,
    training_cfg_path=TRAINING_CFG,
    batch_size=BATCH_SIZE,
    split_ratio=SPLIT_RATIO,
)
full_audit.keys()


## What to look for

- If `bucket_vs_max_time_mismatch` is large, bucket mode is not using the same notion of per-sample circuit length as max-padding preprocessing.
- If `bucket_z_time_is_constant` is `True` for a raw dataset while `gate_length_*` varies a lot, bucket inference is effectively collapsing all samples in that dataset to the configured rectangular tensor length.
- In bucket mode, `bucket_qubits_all_uniform` should be `True`; otherwise the grouping by qubit count is broken.
- The dataloader preview should show bucket mode using loader batch size `1`, with the collate function producing tensors already cut to the bucket-local `z` maxima.
- For the paper SRV setup, the gate pool should stay `['h', 'cx']`, and the per-qubit `min_gates`/`max_gates` should match Extended Data Table I.